In [2]:
today_date = '2025-12-30'

StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 4, Finished, Available, Finished)

In [3]:
abfs_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Files/Landing'

partition_path = f"/Processing_date={today_date}"
complete_path = abfs_path + partition_path
print(complete_path)

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, regexp_replace, to_date

# Define the schema
v_schema = StructType([
    StructField("Row_ID", StringType(), True),
    StructField("Order_ID", StringType(), True),
    StructField("Order_Date", StringType(), True),
    StructField("Ship_Date", StringType(), True),
    StructField("Ship_Mode", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Postal_Code", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Market", StringType(), True),
    StructField("Product_ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub_Category", StringType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True),
    StructField("Shipping_Cost", DoubleType(), True),
    StructField("Order_Priority", StringType(), True),
    StructField("Month", StringType(), True),
    StructField("Year", StringType(), True),
])

# Read CSV
df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(v_schema) \
    .load(complete_path)

# -------------------------------
# DATE TRANSFORMATION (TR → yyyy-MM-dd)
# -------------------------------

# Gün isimlerini kaldır (Pazar, Salı, vb.)
df = df.withColumn(
    "Order_Date",
    regexp_replace(col("Order_Date"), "^[^,]+, ", "")
).withColumn(
    "Ship_Date",
    regexp_replace(col("Ship_Date"), "^[^,]+, ", "")
)

# Türkçe ayları İngilizceye çevir
months = {
    "Ocak": "January",
    "Şubat": "February",
    "Mart": "March",
    "Nisan": "April",
    "Mayıs": "May",
    "Haziran": "June",
    "Temmuz": "July",
    "Ağustos": "August",
    "Eylül": "September",
    "Ekim": "October",
    "Kasım": "November",
    "Aralık": "December"
}

for tr, en in months.items():
    df = df.withColumn("Order_Date", regexp_replace(col("Order_Date"), tr, en)) \
           .withColumn("Ship_Date", regexp_replace(col("Ship_Date"), tr, en))

# String → DateType (yyyy-MM-dd)
df = df.withColumn(
    "Order_Date",
    to_date(col("Order_Date"), "MMMM dd, yyyy")
).withColumn(
    "Ship_Date",
    to_date(col("Ship_Date"), "MMMM dd, yyyy")
)

display(df)
df.createOrReplaceTempView('t_new_data')


StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 5, Finished, Available, Finished)

abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Files/Landing/Processing_date=2025-12-30


SynapseWidget(Synapse.DataFrame, 16338c50-a9db-46eb-8dfa-cee5de0b094d)

In [4]:
%%sql
select * from t_new_data

StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 6, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 26 fields>

In [5]:
Fabric_tblsales_bronze = "abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_bronze"

create_table_sql = None

try:
    spark.read.format("delta") \
        .load(Fabric_tblsales_bronze) \
        .createOrReplaceTempView("t_tblsales_bronze")
except Exception:
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS dbo.tblsales_bronze (
        Row_ID STRING,
        Order_ID STRING,
        Order_Date DATE,
        Ship_Date DATE,
        Ship_Mode STRING,
        Customer_ID STRING,
        Customer_Name STRING,
        Segment STRING,
        Postal_Code STRING,
        City STRING,
        State STRING,
        Country STRING,
        Region STRING,
        Market STRING,
        Product_ID STRING,
        Category STRING,
        Sub_Category STRING,
        Product_Name STRING,
        Sales DOUBLE,
        Quantity INT,
        Discount DOUBLE,
        Profit DOUBLE,
        Shipping_Cost DOUBLE,
        Order_Priority STRING,
        Month STRING,
        Year STRING,
        processing_date DATE
    )
    USING DELTA
    LOCATION 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_bronze'
    """

if create_table_sql:
    spark.sql(create_table_sql)

spark.read.format("delta") \
    .load(Fabric_tblsales_bronze) \
    .createOrReplaceTempView("t_tblsales_bronze")


StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 7, Finished, Available, Finished)

In [6]:
%%sql
select * from t_tblsales_bronze

StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 8, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 27 fields>

In [7]:


sql_statement = f"""MERGE INTO tblsales_bronze as target
                    USING t_new_data as source
                    on target.Order_ID = source.Order_ID and target.Customer_ID = source.Customer_ID

                    WHEN MATCHED THEN
                        UPDATE SET 
                        target.Row_ID = source.Row_ID,
                        target.Order_ID = source.Order_ID,
                        target.Order_Date = source.Order_Date,
                        target.Ship_Date = source.Ship_Date,
                        target.Ship_Mode = source.Ship_Mode,
                        target.Customer_ID = source.Customer_ID,
                        target.Customer_Name = source.Customer_Name,
                        target.Segment = source.Segment,
                        target.Postal_Code = source.Postal_Code,
                        target.City = source.City,
                        target.State = source.State,
                        target.Country = source.Country,
                        target.Region = source.Region,
                        target.Market = source.Market,
                        target.Product_ID = source.Product_ID,
                        target.Category = source.Category,
                        target.Sub_Category = source.Sub_Category,
                        target.Product_Name = source.Product_Name,
                        target.Sales = source.Sales,
                        target.Quantity = source.Quantity,
                        target.Discount = source.Discount,
                        target.Profit = source.Profit,
                        target.Shipping_Cost = source.Shipping_Cost,
                        target.Order_Priority = source.Order_Priority,
                        target.Month = source.Month,
                        target.Year = source.Year,
                        target.processing_date = '{today_date}'

                    WHEN NOT MATCHED THEN
                         INSERT (Row_ID,
                                 Order_ID,
                                 Order_Date,
                                 Ship_Date,
                                 Ship_Mode,
                                 Customer_ID,
                                 Customer_Name,
                                 Segment,
                                 Postal_Code,
                                 City,
                                 State,
                                 Country,
                                 Region,
                                 Market,
                                 Product_ID,
                                 Category,
                                 Sub_Category,
                                 Product_Name,
                                 Sales,
                                 Quantity,
                                 Discount,
                                 Profit,
                                 Shipping_Cost,
                                 Order_Priority,
                                 Month,
                                 Year,
                                 processing_date)
                                 VALUES
                                 (
                                 source.Row_ID,
                                 source.Order_ID,
                                 source.Order_Date,
                                 source.Ship_Date,
                                 source.Ship_Mode,
                                 source.Customer_ID,
                                 source.Customer_Name,
                                 source.Segment,
                                 source.Postal_Code,
                                 source.City,
                                 source.State,
                                 source.Country,
                                 source.Region,
                                 source.Market,
                                 source.Product_ID,
                                 source.Category,
                                 source.Sub_Category,
                                 source.Product_Name,
                                 source.Sales,
                                 source.Quantity,
                                 source.Discount,
                                 source.Profit,
                                 source.Shipping_Cost,
                                 source.Order_Priority,
                                 source.Month,
                                 source.Year,
                                 '{today_date}'
                                 )"""
spark.sql(sql_statement).show()





StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 9, Finished, Available, Finished)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|            14400|               0|               0|            14400|
+-----------------+----------------+----------------+-----------------+



In [8]:
%%sql

select * from tblsales_bronze

StatementMeta(, 293c7fe1-8924-4800-aec3-524fcc0cda9f, 10, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 27 fields>